In [1]:
import torch

print(torch.__version__)
print("CUDA dostępna:", torch.cuda.is_available())
print("Wersja CUDA w PyTorch:", torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.12.0+cu126
CUDA dostępna: True
Wersja CUDA w PyTorch: 12.6
NVIDIA GeForce RTX 4060


In [2]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import librosa
from scipy import signal
from scipy.signal import find_peaks
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_dir = "data/audio_and_txt_files/"
save_crnn_path = "models/best_crnn_model.pth"

SEG_SR = 4000
HOP_LENGTH_SEG = 100
N_MELS_SEG = 64

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def butter_highpass_filter(data, cutoff=50.0, fs=4000, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype="high", analog=False)
    return signal.filtfilt(b, a, data)


# Ten sam patient-wise split co podczas treningu
txt_files = glob.glob(os.path.join(dataset_dir, "*.txt"))

patient_ids = sorted(
    list(set(os.path.basename(f).split("_")[0] for f in txt_files))
)

_, test_patients = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=42
)

test_audio_files = [
    f for f in glob.glob(os.path.join(dataset_dir, "*.wav"))
    if os.path.basename(f).split("_")[0] in test_patients
]

print(f"Liczba nagrań ewaluacyjnych: {len(test_audio_files)}")


class RespiratorySegmentationCRNN(nn.Module):
    def __init__(self, input_channels=1, hidden_size=128):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 1)),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2, 1))
        )

        self.rnn = nn.GRU(
            input_size=512,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        x = self.cnn(x)

        b, c, m, t = x.size()

        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * m)

        x, _ = self.rnn(x)

        return self.classifier(x)


crnn_model = RespiratorySegmentationCRNN().to(device)

crnn_state = torch.load(
    save_crnn_path,
    map_location=device,
    weights_only=False
)

crnn_model.load_state_dict(crnn_state)
crnn_model.eval()

print("✅ Model CRNN załadowany.")


def load_full_recording(wav_path):
    txt_path = wav_path.replace(".wav", ".txt")

    audio, _ = librosa.load(wav_path, sr=SEG_SR)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=SEG_SR,
        n_mels=N_MELS_SEG,
        hop_length=HOP_LENGTH_SEG
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

    gt_intervals = []

    if os.path.exists(txt_path):
        with open(txt_path, "r") as f:
            for line in f:
                parts = line.strip().split("\t")

                if len(parts) >= 2:
                    start = float(parts[0])
                    end = float(parts[1])
                    gt_intervals.append((start, end))

    return mel_db, gt_intervals


def evaluate_boundaries(true_intervals, pred_peaks_time, tolerance_sec=0.5):
    true_boundaries = set()

    for start, end in true_intervals:
        if start > 0:
            true_boundaries.add(start)

    if true_intervals:
        true_boundaries.add(true_intervals[-1][1])

    true_boundaries = sorted(list(true_boundaries))

    TP = 0
    FN = 0
    errors = []
    matched = set()

    for tb in true_boundaries:
        closest_idx = -1
        min_dist = float("inf")

        for i, pb in enumerate(pred_peaks_time):
            if i in matched:
                continue

            d = abs(tb - pb)

            if d < min_dist:
                min_dist = d
                closest_idx = i

        if min_dist <= tolerance_sec:
            TP += 1
            errors.append(min_dist)
            matched.add(closest_idx)
        else:
            FN += 1

    FP = len(pred_peaks_time) - len(matched)

    return TP, FP, FN, errors


print("\nEWALUACJA SEGMENTACJI CRNN")

total_TP = 0
total_FP = 0
total_FN = 0
all_errors = []

with torch.no_grad():
    for i, wav_path in enumerate(test_audio_files, 1):

        print(
            f"\rCRNN: {i}/{len(test_audio_files)} nagrań",
            end="",
            flush=True
        )

        mel_db, gt_intervals = load_full_recording(wav_path)

        input_tensor = (
            torch.tensor(mel_db, dtype=torch.float32)
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        probs = (
            torch.sigmoid(crnn_model(input_tensor))
            .squeeze()
            .cpu()
            .numpy()
        )

        time_axis = np.arange(len(probs)) * (HOP_LENGTH_SEG / SEG_SR)

        probs_smooth = np.convolve(
            probs,
            np.ones(5) / 5,
            mode="same"
        )

        peaks, _ = find_peaks(
            probs_smooth,
            height=0.07,
            distance=40
        )

        pred_peaks_time = [time_axis[p] for p in peaks]

        tp, fp, fn, errs = evaluate_boundaries(
            gt_intervals,
            pred_peaks_time,
            tolerance_sec=0.5
        )

        total_TP += tp
        total_FP += fp
        total_FN += fn
        all_errors.extend(errs)

print()

precision = total_TP / (total_TP + total_FP + 1e-9)
recall = total_TP / (total_TP + total_FN + 1e-9)
f1 = 2 * precision * recall / (precision + recall + 1e-9)
mae = np.mean(all_errors) if all_errors else 0

print("\nWYNIKI CRNN")
print(f"Precyzja: {precision * 100:.2f}%")
print(f"Czułość (Recall): {recall * 100:.2f}%")
print(f"F1-Score: {f1 * 100:.2f}%")
print(f"MAE: {mae:.3f} s")
print(f"TP: {total_TP}, FP: {total_FP}, FN: {total_FN}")

Device: cuda
GPU: NVIDIA GeForce RTX 4060
Liczba nagrań ewaluacyjnych: 214
✅ Model CRNN załadowany.

EWALUACJA SEGMENTACJI CRNN
CRNN: 1/214 nagrań

c:\Users\Admin\Desktop\PracaMGR\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CRNN: 214/214 nagrań

WYNIKI CRNN
Precyzja: 67.82%
Czułość (Recall): 69.81%
F1-Score: 68.80%
MAE: 0.112 s
TP: 1334, FP: 633, FN: 577
